# Monarch · "Powerful Personality" — stickman film (36 s, 9:16)

**Sirf ek tap: `Runtime → Run all`** (Colab par pehle `Runtime → Change runtime type → T4 GPU`).
Kaggle par: `Settings → Accelerator → GPU T4`, `Internet → On`.

Yeh notebook kya karta hai:
1. is repo (`arena/3d8cfad7-monarch-studio`) ko clone karta hai,
2. `tools/kaggle_stick_film.py` chalata hai — 12 locked stickman keyframes se **asli video model**
   (Wan 2.2 image-to-video; T4 par na chale to LTX-Video 2B) se clips banata hai,
3. un clips par **captions + voice + sound** laga kar 1080×1920 master jodta hai,
4. video yahin notebook me dikhata hai — `monarch_stick_36s.mp4`,
5. saath hi **launch kit** banata hai: 11s teaser (Reels/Shorts), cover/thumbnail, `.srt` subtitles,
   12-scene contact sheet, aur copy-paste caption (`kit.md`).

Session toot jaye to dobara `Run all`: jo scene ban chuki hai, woh skip ho jayegi.


### 1 · Setup (repo clone)

In [ ]:
#@title 1 · Repo clone / update
import pathlib, subprocess

BRANCH = "arena/3d8cfad7-monarch-studio"
REPO = "https://github.com/adil-chandio/Monarch-Studio"
base = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else pathlib.Path("/content")
repo = base / "repo"

if (repo / "tools/kaggle_stick_film.py").exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(repo)], check=True)

head = subprocess.run(["git", "-C", str(repo), "log", "--oneline", "-1"],
                      capture_output=True, text=True).stdout.strip()
print("repo:", repo)
print("commit:", head)


### 2 · GPU check (yeh pass hona chahiye)

In [ ]:
#@title 2 · GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise SystemExit("GPU on karo, phir Run all: Colab Runtime > Change runtime type > T4 GPU "
                     "| Kaggle Settings > Accelerator > GPU T4")
p = torch.cuda.get_device_properties(0)
print("VRAM:", round(p.total_memory / 1e9, 1), "GB | bf16:", torch.cuda.is_bf16_supported())


### 3 · Film banao (12 scenes — sabse lamba step)

Pehla run me models download honge (Wan ~10 GB) — 6–10 min lag sakte hain. Uske baad har scene
ka apna time. Total 2–4 ghante (GPU par depend). Beech me session toot jaye to cell dobara chala do.


In [ ]:
#@title 3 · Render 12 scenes + master
import subprocess, sys

out = base / "stick"
cmd = [sys.executable, str(repo / "tools/kaggle_stick_film.py"),
       "--out", str(out), "--final", str(base)]
print(" ".join(cmd), flush=True)
proc = subprocess.run(cmd)
print("exit code:", proc.returncode)


### 3b · (optional) Sirf ek scene ka test — pehle yeh chala kar tasalli

Pehli dafa poora run karne se pehle ek scene dekh lo (~10–15 min). Theek lage to **cell 3** chala do.


In [ ]:
#@title 3b · Sirf ek scene (optional)
import subprocess, sys

scene = "k01"          # k01..k12
cmd = [sys.executable, str(repo / "tools/kaggle_stick_film.py"),
       "--out", str(base / "stick"), "--final", str(base / "test"), "--only", scene]
print(" ".join(cmd), flush=True)
subprocess.run(cmd)


### 4 · Film + launch kit dekho

Master ke saath kit bhi banti hai: **teaser** (11s hook cut), **cover** (thumbnail), **subtitles.srt**,
**contact sheet** aur **kit.md** (caption + hashtags + beat sheet).

In [ ]:
#@title 4 · Master + teaser + cover dikhao
import pathlib
from IPython.display import Image, Video, display

base = pathlib.Path(base)
master = base / "monarch_stick_36s.mp4"
kit = base / "kit"

if master.exists():
    print("MASTER:", master.name, round(master.stat().st_size / 1e6, 1), "MB")
    display(Video(str(master), embed=False, width=420))
else:
    print("master nahi bana — cell 3 chalao")

teaser = kit / "teaser_11s.mp4"
if teaser.exists():
    print("TEASER:", teaser.name, round(teaser.stat().st_size / 1e6, 1), "MB")
    display(Video(str(teaser), embed=False, width=320))

cover = kit / "cover.jpg"
if cover.exists():
    display(Image(filename=str(cover), width=320))

if (kit / "kit.md").exists():
    print((kit / "kit.md").read_text()[:1200])


### 5 · (Colab) Drive me save — film + kit

In [ ]:
#@title 5 · Drive me copy (Colab)
import pathlib, shutil

master = pathlib.Path(base) / "monarch_stick_36s.mp4"
if not master.exists():
    print("master nahi bana — pehle cell 3 chalao")
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dst = pathlib.Path('/content/drive/MyDrive/Monarch-T2V'); dst.mkdir(parents=True, exist_ok=True)
        (dst / "kit").mkdir(exist_ok=True)
        shutil.copy2(master, dst / master.name)
        kit = pathlib.Path(base) / "kit"
        for f in kit.glob("*") if kit.exists() else []:
            shutil.copy2(f, dst / "kit" / f.name)
        receipts = pathlib.Path(base) / "stick" / "receipts.json"
        if receipts.exists():
            shutil.copy2(receipts, dst / "stick_receipts.json")
        print("Drive me:", dst / master.name, "+ kit/")
    except Exception as exc:
        print("Drive copy skip:", type(exc).__name__, exc)
